# NETRA-X &mdash; Project Notebook

**A confidence-scored entity-resolution system for dark-web threat-actor de-anonymization.**

Smart India Hackathon 2026 &middot; Problem Statement **SIH26151** &middot; sponsored by the
National Technical Research Organisation (NTRO) &middot; Software track, Blockchain &amp; Cybersecurity theme.

---

This notebook is the single reference for the project: what it does, how it is built,
what the attribution mathematics actually computes, what is finished versus planned,
and where the gaps are. Every number in the engine sections was produced by running
the code in this repository &mdash; nothing here is illustrative or approximated.

| | |
|---|---|
| **Repository** | `github.com/me13krishna/NETRA-X` (`main`) |
| **Language** | Python 3.11+ backend, TypeScript frontend |
| **Team** | 6 members across 3 pairs |
| **Status** | Phase 0 complete, Phases 1&ndash;3 active |

## 1. The problem

NTRO's brief asks for a system that de-anonymizes dark-web threat actors and links them
to suspect real-world entities, through three required capabilities:

1. **Find misconfigurations in Tor hidden services** &mdash; exposed status pages, TLS
   certificates tied to clearnet domains, default banners, descriptor inconsistencies &mdash;
   and match them to clearnet infrastructure.
2. **Map threat actors across multiple marketplaces** into a single relationship graph of
   handles, PGP keys, wallets, and trust links.
3. **Use stylometric and behavioral analysis** to link rebranded or migrated personas to
   known threat actors.

The system must run autonomously, expose a queryable analytical front end, and export
results as CSV, JSON, and report formats.

### What NETRA-X actually does

It ingests scattered fragments about anonymous personas &mdash; forum posts, marketplace
listings, PGP keys, wallet addresses, hidden-service metadata &mdash; and determines which
fragments belong to the same real actor, **even when that actor deliberately hides behind
multiple throwaway identities.**

It does not attack Tor. It correlates what operators already leaked: a reused certificate,
a reused wallet, a writing style that does not change when the username does.

An investigator searches a handle, wallet address, or PGP fingerprint and gets back a graph
of every other persona that is provably or probably the same actor &mdash; **each link
labelled with its evidence and a confidence score, never a black-box guess.** Where the
evidence is not strong enough, the system stays silent rather than force an accusation.

## 2. System architecture

Five stages. Evidence flows left to right; every stage writes provenance into the
persistence layer beneath it, so any score can be traced back to the raw artifact that
produced it.

<div style="margin:6px 0 2px"><svg xmlns="http://www.w3.org/2000/svg" width="880" height="486" viewBox="0 0 880 486" role="img" aria-label="NETRA-X system architecture, five stages from collection to delivery"><rect width="880" height="486" fill="#fcfcfb" rx="6"/><style>text{font-family:ui-sans-serif,-apple-system,Segoe UI,Roboto,Helvetica,Arial,sans-serif}.ti{font-size:13px;font-weight:650;fill:#0b0b0b}.su{font-size:11px;fill:#52514e}.lb{font-size:11.5px;fill:#0b0b0b;font-weight:600}.sb{font-size:10px;fill:#8a8984}.vl{font-family:ui-monospace,SFMono-Regular,Cascadia Mono,Consolas,monospace;font-size:11px;fill:#0b0b0b;font-weight:600}.no{font-size:10px;fill:#52514e}.hd{font-size:10px;fill:#8a8984;letter-spacing:.09em;font-weight:700}.mo{font-family:ui-monospace,SFMono-Regular,Cascadia Mono,Consolas,monospace;font-size:9.5px;fill:#52514e}.ax{font-family:ui-monospace,SFMono-Regular,Cascadia Mono,Consolas,monospace;font-size:9.5px;fill:#8a8984}</style><rect x="24" y="54" width="136" height="158" rx="7" fill="#ffffff" stroke="#dedcd5"/><text class="hd" x="36" y="44">01  COLLECT</text><rect x="34" y="66" width="116" height="36" rx="5" fill="#f7f6f2"/><text class="lb" x="42" y="80" style="font-size:10.5px">Onion pages, forums</text><text class="mo" x="42" y="94">onion_probe.py</text><rect x="34" y="112" width="116" height="36" rx="5" fill="#f7f6f2"/><text class="lb" x="42" y="126" style="font-size:10.5px">WARC + SHA-256</text><text class="mo" x="42" y="140">warc_writer.py</text><rect x="34" y="158" width="116" height="36" rx="5" fill="#f7f6f2"/><text class="lb" x="42" y="172" style="font-size:10.5px">Redis Streams bus</text><text class="mo" x="42" y="186">event_bus.py</text><rect x="198" y="54" width="136" height="204" rx="7" fill="#ffffff" stroke="#dedcd5"/><text class="hd" x="210" y="44">02  EXTRACT</text><rect x="208" y="66" width="116" height="36" rx="5" fill="#f7f6f2"/><text class="lb" x="216" y="80" style="font-size:10.5px">PGP / BTC / ETH / XMR</text><text class="mo" x="216" y="94">extractor.py</text><rect x="208" y="112" width="116" height="36" rx="5" fill="#f7f6f2"/><text class="lb" x="216" y="126" style="font-size:10.5px">Favicon mmh3, TLS</text><text class="mo" x="216" y="140">onion_probe.py</text><rect x="208" y="158" width="116" height="36" rx="5" fill="#f7f6f2"/><text class="lb" x="216" y="172" style="font-size:10.5px">SimHash clone detect</text><text class="mo" x="216" y="186">clone_detector.py</text><rect x="208" y="204" width="116" height="36" rx="5" fill="#f7f6f2"/><text class="lb" x="216" y="218" style="font-size:10.5px">Stylometry features</text><text class="mo" x="216" y="232">packages/stylometry</text><rect x="372" y="54" width="136" height="250" rx="7" fill="#ffffff" stroke="#dedcd5"/><text class="hd" x="384" y="44">03  FUSE</text><rect x="382" y="66" width="116" height="36" rx="5" fill="#f7f6f2"/><text class="lb" x="390" y="80" style="font-size:10.5px">LLR per evidence item</text><text class="mo" x="390" y="94">fusion.py</text><rect x="382" y="112" width="116" height="36" rx="5" fill="#f7f6f2"/><text class="lb" x="390" y="126" style="font-size:10.5px">Dependence discount</text><text class="mo" x="390" y="140">lambda = 0.25</text><rect x="382" y="158" width="116" height="36" rx="5" fill="#f7f6f2"/><text class="lb" x="390" y="172" style="font-size:10.5px">Family caps</text><text class="mo" x="390" y="186">common/types.py</text><rect x="382" y="204" width="116" height="36" rx="5" fill="#f7f6f2"/><text class="lb" x="390" y="218" style="font-size:10.5px">Contradiction penalty</text><text class="mo" x="390" y="232">mu_table.yaml</text><rect x="382" y="250" width="116" height="36" rx="5" fill="#f7f6f2"/><text class="lb" x="390" y="264" style="font-size:10.5px">Calibrate to P</text><text class="mo" x="390" y="278">calibration.py</text><rect x="546" y="54" width="136" height="204" rx="7" fill="#ffffff" stroke="#dedcd5"/><text class="hd" x="558" y="44">04  DECIDE</text><rect x="556" y="66" width="116" height="36" rx="5" fill="#f7f6f2"/><text class="lb" x="564" y="80" style="font-size:10.5px">Threshold ladder</text><text class="mo" x="564" y="94">decide.py</text><rect x="556" y="112" width="116" height="36" rx="5" fill="#f7f6f2"/><text class="lb" x="564" y="126" style="font-size:10.5px">Hypothesis + waterfall</text><text class="mo" x="564" y="140">PostgreSQL 16</text><rect x="556" y="158" width="116" height="36" rx="5" fill="#f7f6f2"/><text class="lb" x="564" y="172" style="font-size:10.5px">Graph projection</text><text class="mo" x="564" y="186">Neo4j 5</text><rect x="556" y="204" width="116" height="36" rx="5" fill="#f7f6f2"/><text class="lb" x="564" y="218" style="font-size:10.5px">Hash-chained audit</text><text class="mo" x="564" y="232">evidence/audit.py</text><rect x="720" y="54" width="136" height="204" rx="7" fill="#ffffff" stroke="#dedcd5"/><text class="hd" x="732" y="44">05  DELIVER</text><rect x="730" y="66" width="116" height="36" rx="5" fill="#f7f6f2"/><text class="lb" x="738" y="80" style="font-size:10.5px">Review Queue</text><text class="mo" x="738" y="94">Next.js 14</text><rect x="730" y="112" width="116" height="36" rx="5" fill="#f7f6f2"/><text class="lb" x="738" y="126" style="font-size:10.5px">Evidence Waterfall</text><text class="mo" x="738" y="140">React 18</text><rect x="730" y="158" width="116" height="36" rx="5" fill="#f7f6f2"/><text class="lb" x="738" y="172" style="font-size:10.5px">Graph Explorer</text><text class="mo" x="738" y="186">Cytoscape.js</text><rect x="730" y="204" width="116" height="36" rx="5" fill="#f7f6f2"/><text class="lb" x="738" y="218" style="font-size:10.5px">PDF / STIX 2.1 / CSV</text><text class="mo" x="738" y="232">reporting.py</text><path d="M168 150 l14 0 m-5 -5 l5 5 l-5 5" stroke="#2a78d6" stroke-width="1.6" fill="none" stroke-linecap="round" stroke-linejoin="round"/><path d="M342 150 l14 0 m-5 -5 l5 5 l-5 5" stroke="#2a78d6" stroke-width="1.6" fill="none" stroke-linecap="round" stroke-linejoin="round"/><path d="M516 150 l14 0 m-5 -5 l5 5 l-5 5" stroke="#2a78d6" stroke-width="1.6" fill="none" stroke-linecap="round" stroke-linejoin="round"/><path d="M690 150 l14 0 m-5 -5 l5 5 l-5 5" stroke="#2a78d6" stroke-width="1.6" fill="none" stroke-linecap="round" stroke-linejoin="round"/><rect x="24" y="404" width="832" height="58" rx="7" fill="#f2f6fc" stroke="#d6e4f7"/><text class="hd" x="40" y="428" style="fill:#2a78d6">PERSISTENCE AND PROVENANCE</text><text class="mo" x="40" y="450" style="fill:#0b0b0b">PostgreSQL 16 + pgvector</text><text class="mo" x="208" y="450" style="fill:#0b0b0b">Neo4j 5</text><text class="mo" x="376" y="450" style="fill:#0b0b0b">Redis 7</text><text class="mo" x="544" y="450" style="fill:#0b0b0b">MinIO</text><text class="mo" x="712" y="450" style="fill:#0b0b0b">OpenSearch</text></svg></div>

<div style="font-size:12px;color:#8a8984;font-family:ui-sans-serif,system-ui,sans-serif;margin-bottom:14px">Figure 1 &mdash; The five-stage pipeline. File names are the actual modules in this repository.</div>

### Why a modular monolith

The backend is a single FastAPI application rather than a set of microservices. For a
six-person team on a hackathon clock, one deployable unit with clean package boundaries
(`packages/attribution`, `packages/stylometry`, `packages/evidence`, `packages/graph`)
gives the separation benefits without the operational cost. The package boundaries are
real &mdash; `apps/api` imports the engine only through `packages/evidence/attribution.py`,
a deliberate bridge module, so the engine can be restructured without touching the API.

## 3. Technology stack

Verified against the repository on 30 August 2026. The status column is deliberately
honest &mdash; it distinguishes what runs today from what is scaffolded or roadmapped.

| Layer | Technology | Status |
|---|---|---|
| **Backend** | Python 3.11+, FastAPI, Uvicorn | Running &mdash; 24 REST routes |
| | SQLAlchemy 2 (sync + async), Alembic | Running |
| | Pydantic v2, pydantic-settings | Running |
| | PyJWT, passlib/bcrypt | Running &mdash; JWT auth |
| **Frontend** | Next.js 14.1.4 (App Router), React 18 | Running |
| | TypeScript 5.4, Tailwind CSS 3.4 | Running |
| | Cytoscape.js 3.28 | Running &mdash; graph explorer |
| | Framer Motion, TanStack Query | Running |
| **Primary store** | PostgreSQL 16 + pgvector | Running (SQLite fallback for local dev) |
| **Graph** | Neo4j 5.18 | Code complete; falls back to relational topology when offline |
| **Cache / bus** | Redis 7 | Scaffolded &mdash; `event_bus.py` only, Phase 2 |
| **Object store** | MinIO | Declared; **no code path yet**, Phase 2 |
| **Search** | OpenSearch 2.12 | In compose; **not yet wired**, Phase 4 |
| **Analysis** | scikit-learn, NumPy, SciPy, NetworkX | Running |
| | mmh3 (favicon hashing) | Running |
| | ImageHash, OpenCV, exifread | Installed, visual forensics pending |
| **Reporting** | ReportLab (PDF), STIX 2.1, CSV | Running |
| **Ops** | Docker, Docker Compose, Makefile, Render | Compose has one broken build path (see &sect;10) |

> **Note on the pitch deck.** The Technical Approach slide shows *GitHub Actions* and
> *SSDEEP*. Neither exists in the repository or the roadmap. The slide also labels
> calibration as *isotonic*; the live path is a **sigmoid**. See &sect;10.

## 4. Repository layout

```
NETRA-X/
├─ apps/
│  ├─ api/                    FastAPI modular monolith
│  │  ├─ main.py              24 REST routes, auth, exports
│  │  └─ database/            SQLAlchemy models + session
│  └─ web/                    Next.js 14 App Router
│     └─ src/components/      13 React components
├─ packages/
│  ├─ attribution/            THE CORE ENGINE
│  │  ├─ fusion.py            LLR fusion, dependence discount, family caps
│  │  ├─ calibration.py       Sigmoid + isotonic calibrators
│  │  ├─ candidate_gen.py     Pair candidate generation
│  │  ├─ decide.py            Threshold ladder
│  │  └─ mu_table.yaml        m/u priors for every feature
│  ├─ stylometry/             Char n-grams, function words, punctuation
│  ├─ evidence/               Audit chain, auth, reporting, STIX export
│  ├─ graph/                  Neo4j projection
│  ├─ schemas/                Pydantic contracts
│  └─ common/                 Shared types, family caps
├─ workers/
│  ├─ collection/             WARC writer, onion probe, Redis event bus
│  └─ extraction/             Identifier extractors, SimHash clone detector
├─ bench/                     Calibration metrics (ECE, Brier, FAR, Recall@10)
├─ seed/                      Synthetic ground-truth generators
├─ tests/                     39 tests
└─ docs/                      Architecture, roadmap, role docs
```

## 5. The attribution engine

This is the differentiator. Everything else in the project is plumbing that feeds it or
surfaces its output.

The engine follows the **Fellegi-Sunter probabilistic record-linkage model** (1969), not
raw graph connectivity. The difference matters: a naive system says *"they share a wallet,
therefore same person."* NETRA-X asks *"how surprising would this coincidence be if they
were strangers?"* &mdash; and weighs every clue by that answer.

### 5.1 The log-likelihood ratio

Every feature carries two probabilities, stored in
[`packages/attribution/mu_table.yaml`](packages/attribution/mu_table.yaml):

- **m** = probability of observing this clue **if the two personas are the same actor**
- **u** = probability of observing it **by pure coincidence between strangers**

$$\text{LLR}_i = \ln\left(\frac{m_i}{u_i}\right)$$

Logarithms let independent evidence be **added** instead of multiplied, which is what makes
the whole scheme tractable and explainable. Each item's contribution is further scaled by
source reliability and a credibility multiplier before it enters the sum.

Here are the real priors and their resulting LLRs, read straight from the table:

In [1]:
import math
from packages.attribution.fusion import load_mu_table

mu = load_mu_table()

print(f"{'FEATURE':<32} {'FAMILY':<17} {'m':>8} {'u':>10} {'LLR':>7}")
print("-" * 78)
for name, f in mu["features"].items():
    llr = math.log(f["m_i"] / f["u_i"])
    print(f"{name:<32} {f['family']:<17} {f['m_i']:>8} {f['u_i']:>10} {llr:>7.2f}")

print()
print("CONTRADICTIONS (subtract, and are never capped)")
print("-" * 78)
for name, c in mu["contradictions"].items():
    print(f"{name:<32} {'':<17} {'':>8} {'':>10} {-c['contradiction_weight']:>7.1f}")

FEATURE                          FAMILY                   m          u     LLR
------------------------------------------------------------------------------
pgp_fingerprint_exact            EXACT_IDENTITY      0.9999      1e-08   18.42
ssh_host_key_fingerprint         EXACT_IDENTITY       0.999      1e-07   16.12
btc_address_reuse                FINANCIAL             0.95      1e-05   11.46
btc_co_input_clustering          FINANCIAL              0.9     0.0001    9.10
favicon_mmh3_hash                INFRASTRUCTURE        0.92     0.0001    9.13
ssl_tls_cert_serial              INFRASTRUCTURE        0.88     0.0005    7.47
simhash_clone_95                 CONTENT_NLP           0.85      0.001    6.75
unique_typo_signature            CONTENT_NLP           0.75      0.005    5.01
stylometry_burrows_delta         STYLOMETRY            0.82       0.01    4.41
temporal_diurnal_fit             TEMPORAL               0.7        0.1    1.95
handle_trigram_fuzzy             SEMANTIC_HANDLE    

Read the spread. A **PGP fingerprint match scores 18.42** because private keys
essentially never collide by accident (`u = 1e-08`). A **shared posting timezone scores
1.95** because millions of people post in the same hours (`u = 0.10`). The engine encodes
investigative judgement as arithmetic.

### 5.2 Dependence discounting &mdash; lambda = 0.25

**The problem:** two clues can be the same clue wearing a different hat.

`btc_address_reuse` and `btc_co_input_clustering` both sit in the dependence group
`wallet_cluster_btc`. Adding them naively counts *one* wallet leak twice and inflates
confidence on evidence that is not actually independent.

**The fix:** within a dependence group, the strongest item counts in full and every
additional item is multiplied by **lambda = 0.25** &mdash; 25% credit.

This is the single most important honesty mechanism in the engine. Without it, an actor
who leaks one wallet in five observable ways looks five times as identified as they are.

### 5.3 Family caps

Even after discounting, one *type* of evidence should not be able to run away with the
verdict. Each family carries a hard ceiling, defined in
[`packages/common/types.py`](packages/common/types.py):

<div style="margin:6px 0 2px"><svg xmlns="http://www.w3.org/2000/svg" width="880" height="380" viewBox="0 0 880 380" role="img" aria-label="Maximum LLR contribution allowed per evidence family"><rect width="880" height="380" fill="#fcfcfb" rx="6"/><style>text{font-family:ui-sans-serif,-apple-system,Segoe UI,Roboto,Helvetica,Arial,sans-serif}.ti{font-size:13px;font-weight:650;fill:#0b0b0b}.su{font-size:11px;fill:#52514e}.lb{font-size:11.5px;fill:#0b0b0b;font-weight:600}.sb{font-size:10px;fill:#8a8984}.vl{font-family:ui-monospace,SFMono-Regular,Cascadia Mono,Consolas,monospace;font-size:11px;fill:#0b0b0b;font-weight:600}.no{font-size:10px;fill:#52514e}.hd{font-size:10px;fill:#8a8984;letter-spacing:.09em;font-weight:700}.mo{font-family:ui-monospace,SFMono-Regular,Cascadia Mono,Consolas,monospace;font-size:9.5px;fill:#52514e}.ax{font-family:ui-monospace,SFMono-Regular,Cascadia Mono,Consolas,monospace;font-size:9.5px;fill:#8a8984}</style><text class="ti" x="24" y="28">Family caps &#8212; the ceiling on any single evidence type</text><text class="su" x="24" y="46">No one kind of evidence can carry a verdict alone. Values are maximum LLR contribution.</text><line x1="250.0" y1="62" x2="250.0" y2="350" stroke="#dedcd5" stroke-width="1"/><text class="ax" x="250.0" y="366" text-anchor="middle">0</text><line x1="344.0" y1="62" x2="344.0" y2="350" stroke="#dedcd5" stroke-width="1"/><text class="ax" x="344.0" y="366" text-anchor="middle">2</text><line x1="438.0" y1="62" x2="438.0" y2="350" stroke="#dedcd5" stroke-width="1"/><text class="ax" x="438.0" y="366" text-anchor="middle">4</text><line x1="532.0" y1="62" x2="532.0" y2="350" stroke="#dedcd5" stroke-width="1"/><text class="ax" x="532.0" y="366" text-anchor="middle">6</text><line x1="626.0" y1="62" x2="626.0" y2="350" stroke="#dedcd5" stroke-width="1"/><text class="ax" x="626.0" y="366" text-anchor="middle">8</text><line x1="720.0" y1="62" x2="720.0" y2="350" stroke="#dedcd5" stroke-width="1"/><text class="ax" x="720.0" y="366" text-anchor="middle">10</text><text class="lb" x="24" y="91">Exact identity</text><text class="sb" x="24" y="105">PGP / SSH key fingerprint</text><rect x="250" y="80" width="470" height="20" rx="4" fill="#eceae3"/><path d="M250 80 h466.0 a4 4 0 0 1 4 4 v12 a4 4 0 0 1 -4 4 h-466.0 z" fill="#2a78d6"/><text class="vl" x="730.0" y="95">10</text><text class="lb" x="24" y="131">Financial</text><text class="sb" x="24" y="145">Wallet reuse, co-input cluster</text><rect x="250" y="120" width="470" height="20" rx="4" fill="#eceae3"/><path d="M250 120 h348.5 a4 4 0 0 1 4 4 v12 a4 4 0 0 1 -4 4 h-348.5 z" fill="#2a78d6"/><text class="vl" x="612.5" y="135">7.5</text><text class="lb" x="24" y="171">Infrastructure</text><text class="sb" x="24" y="185">TLS cert serial, favicon mmh3</text><rect x="250" y="160" width="470" height="20" rx="4" fill="#eceae3"/><path d="M250 160 h231.0 a4 4 0 0 1 4 4 v12 a4 4 0 0 1 -4 4 h-231.0 z" fill="#2a78d6"/><text class="vl" x="495.0" y="175">5</text><text class="lb" x="24" y="211">Content / NLP</text><text class="sb" x="24" y="225">SimHash clone, typo signature</text><rect x="250" y="200" width="470" height="20" rx="4" fill="#eceae3"/><path d="M250 200 h231.0 a4 4 0 0 1 4 4 v12 a4 4 0 0 1 -4 4 h-231.0 z" fill="#2a78d6"/><text class="vl" x="495.0" y="215">5</text><text class="lb" x="24" y="251">Stylometry</text><text class="sb" x="24" y="265">Burrows Delta, function words</text><rect x="250" y="240" width="470" height="20" rx="4" fill="#eceae3"/><path d="M250 240 h137.0 a4 4 0 0 1 4 4 v12 a4 4 0 0 1 -4 4 h-137.0 z" fill="#2a78d6"/><text class="vl" x="401.0" y="255">3</text><text class="lb" x="24" y="291">Temporal</text><text class="sb" x="24" y="305">Diurnal posting overlap</text><rect x="250" y="280" width="470" height="20" rx="4" fill="#eceae3"/><path d="M250 280 h90.0 a4 4 0 0 1 4 4 v12 a4 4 0 0 1 -4 4 h-90.0 z" fill="#2a78d6"/><text class="vl" x="354.0" y="295">2</text><text class="lb" x="24" y="331">Semantic handle</text><text class="sb" x="24" y="345">Handle trigram similarity</text><rect x="250" y="320" width="470" height="20" rx="4" fill="#eceae3"/><path d="M250 320 h90.0 a4 4 0 0 1 4 4 v12 a4 4 0 0 1 -4 4 h-90.0 z" fill="#2a78d6"/><text class="vl" x="354.0" y="335">2</text></svg></div>

<div style="font-size:12px;color:#8a8984;font-family:ui-sans-serif,system-ui,sans-serif;margin-bottom:14px">Figure 2 &mdash; Family caps. Stylometry is capped at 3.0 because it is adversarially defeatable: anyone who knows to change their writing can beat it.</div>

The consequence is structural: **you cannot reach a confident verdict from one family
alone.** A PGP match alone scores 10.0 after capping. Reaching high confidence requires
*independent kinds* of evidence to agree &mdash; which is exactly how a competent
investigator reasons.

### 5.4 Contradictions

Evidence that argues *against* a match subtracts, and &mdash; importantly &mdash;
**is never capped**:

| Contradiction | Weight |
|---|---|
| `pgp_key_conflict` &mdash; conflicting keys published for the same profile | **&minus;20.0** |
| `temporal_impossible_overlap` &mdash; simultaneous posts from opposing geolocations | **&minus;15.0** |

The asymmetry is deliberate. One solid disproof should be able to kill a large pile of
weak circumstantial agreement. Capping the positives but not the negatives builds
skepticism into the arithmetic.

### 5.5 Calibration and the decision ladder

A final LLR of 27.45 means nothing to an analyst, so it is mapped to a probability:

$$P(H_1 \mid E) = \frac{1}{1 + e^{-(\text{LLR} + \text{prior})}}, \qquad \text{prior} = -2.0$$

The &minus;2.0 prior encodes the correct default posture: *before seeing any evidence,
assume two random personas are probably **not** the same person.* The engine has to be
talked out of skepticism.

> **Honest note.** `IsotonicCalibrator` exists in `calibration.py` and is the intended
> Phase 1 destination, but nothing currently fits it &mdash; isotonic regression needs
> labelled ground-truth pairs first. The live path is the sigmoid above. The code says so
> itself in `packages/evidence/attribution.py:191`.

<div style="margin:6px 0 2px"><svg xmlns="http://www.w3.org/2000/svg" width="880" height="216" viewBox="0 0 880 216" role="img" aria-label="Decision threshold ladder"><rect width="880" height="216" fill="#fcfcfb" rx="6"/><style>text{font-family:ui-sans-serif,-apple-system,Segoe UI,Roboto,Helvetica,Arial,sans-serif}.ti{font-size:13px;font-weight:650;fill:#0b0b0b}.su{font-size:11px;fill:#52514e}.lb{font-size:11.5px;fill:#0b0b0b;font-weight:600}.sb{font-size:10px;fill:#8a8984}.vl{font-family:ui-monospace,SFMono-Regular,Cascadia Mono,Consolas,monospace;font-size:11px;fill:#0b0b0b;font-weight:600}.no{font-size:10px;fill:#52514e}.hd{font-size:10px;fill:#8a8984;letter-spacing:.09em;font-weight:700}.mo{font-family:ui-monospace,SFMono-Regular,Cascadia Mono,Consolas,monospace;font-size:9.5px;fill:#52514e}.ax{font-family:ui-monospace,SFMono-Regular,Cascadia Mono,Consolas,monospace;font-size:9.5px;fill:#8a8984}</style><text class="ti" x="24" y="28">Decision ladder &#8212; packages/attribution/decide.py</text><text class="su" x="24" y="46">An active contradiction short-circuits the ladder before probability is ever consulted.</text><rect x="24" y="64" width="4" height="30" rx="2" fill="#e34948"/><text class="vl" x="40" y="77" style="fill:#e34948">CONTRADICTION_REJECTED</text><text class="no" x="40" y="91">contradiction penalty &gt; 0, or final LLR &lt; 0</text><text class="no" x="700" y="84" style="fill:#8a8984">checked first</text><rect x="24" y="100" width="4" height="30" rx="2" fill="#1baf7a"/><text class="vl" x="40" y="113" style="fill:#1baf7a">HIGH_CONFIDENCE_LINK</text><text class="no" x="40" y="127">posterior P &#8805; 0.85</text><text class="no" x="700" y="120" style="fill:#8a8984">analyst confirms</text><rect x="24" y="136" width="4" height="30" rx="2" fill="#2a78d6"/><text class="vl" x="40" y="149" style="fill:#2a78d6">LOW_CONFIDENCE_LINK</text><text class="no" x="40" y="163">0.50 &#8804; P &lt; 0.85</text><text class="no" x="700" y="156" style="fill:#8a8984">queued for review</text><rect x="24" y="172" width="4" height="30" rx="2" fill="#8a8984"/><text class="vl" x="40" y="185" style="fill:#8a8984">INSUFFICIENT_EVIDENCE</text><text class="no" x="40" y="199">P &lt; 0.50</text><text class="no" x="700" y="192" style="fill:#8a8984">no claim is made</text></svg></div>

<div style="font-size:12px;color:#8a8984;font-family:ui-sans-serif,system-ui,sans-serif;margin-bottom:14px">Figure 3 &mdash; The decision ladder. Contradictions are evaluated before probability, so a disproved pair is rejected outright rather than scored.</div>

## 6. Worked example &mdash; running the real engine

Two personas share a PGP key, a Bitcoin address, a co-input wallet cluster, a favicon
hash, a writing style, and a posting rhythm. Six clues across five families.

In [2]:
from packages.attribution.fusion import LLRFusionEngine
from packages.attribution.decide import evaluate_attribution

engine = LLRFusionEngine()          # lambda = 0.25 by default

def item(eid, feature):
    return engine.create_item_from_prior(eid, feature)

evidence = [
    item("E1", "pgp_fingerprint_exact"),
    item("E2", "btc_address_reuse"),
    item("E3", "btc_co_input_clustering"),   # same dependence group as E2
    item("E4", "favicon_mmh3_hash"),
    item("E5", "stylometry_burrows_delta"),
    item("E6", "temporal_diurnal_fit"),
]

result = evaluate_attribution(evidence)

print(f"{'ITEM':<30} {'RAW':>7} {'COUNTED':>9}  NOTE")
print("-" * 72)
for c in result.contributions:
    note = []
    if c["is_discounted"]:
        note.append("lambda x0.25")
    if c["is_capped"]:
        note.append("family cap")
    print(f"{c['feature_name']:<30} {c['raw_llr']:>7.2f} {c['llr_contrib']:>9.2f}  "
          f"{', '.join(note) or 'counted in full'}")

raw_total = sum(c["raw_llr"] for c in result.contributions)
print("-" * 72)
print(f"{'raw sum (naive)':<30} {raw_total:>7.2f}")
print(f"{'final LLR (engine)':<30} {'':>7} {result.final_llr:>9.2f}")
print(f"{'discarded as correlated/over-cap':<30} {'':>7} {raw_total - result.final_llr:>9.2f}")
print()
print("posterior P :", round(result.posterior_probability, 8))
print("decision    :", result.decision.value)
print("families    :", result.independent_family_count, "independent ->", ", ".join(result.families_present))

ITEM                               RAW   COUNTED  NOTE
------------------------------------------------------------------------
pgp_fingerprint_exact            18.42     10.00  family cap
btc_address_reuse                11.46      6.26  family cap
btc_co_input_clustering           9.11      1.24  lambda x0.25, family cap
favicon_mmh3_hash                 9.13      5.00  family cap
stylometry_burrows_delta          4.41      3.00  family cap
temporal_diurnal_fit              1.95      1.95  counted in full
------------------------------------------------------------------------
raw sum (naive)                  54.47
final LLR (engine)                         27.45
discarded as correlated/over-cap             27.02

posterior P : 1.0
decision    : HIGH_CONFIDENCE_LINK
families    : 5 independent -> EXACT_IDENTITY, FINANCIAL, INFRASTRUCTURE, STYLOMETRY, TEMPORAL


A naive additive system would have reported **54.48**. The engine reports **27.45** &mdash;
it threw away exactly half the apparent evidence as correlated or over-cap. That gap is
the product.

<div style="margin:6px 0 2px"><svg xmlns="http://www.w3.org/2000/svg" width="880" height="444" viewBox="0 0 880 444" role="img" aria-label="Evidence waterfall showing raw versus counted LLR for each item"><rect width="880" height="444" fill="#fcfcfb" rx="6"/><style>text{font-family:ui-sans-serif,-apple-system,Segoe UI,Roboto,Helvetica,Arial,sans-serif}.ti{font-size:13px;font-weight:650;fill:#0b0b0b}.su{font-size:11px;fill:#52514e}.lb{font-size:11.5px;fill:#0b0b0b;font-weight:600}.sb{font-size:10px;fill:#8a8984}.vl{font-family:ui-monospace,SFMono-Regular,Cascadia Mono,Consolas,monospace;font-size:11px;fill:#0b0b0b;font-weight:600}.no{font-size:10px;fill:#52514e}.hd{font-size:10px;fill:#8a8984;letter-spacing:.09em;font-weight:700}.mo{font-family:ui-monospace,SFMono-Regular,Cascadia Mono,Consolas,monospace;font-size:9.5px;fill:#52514e}.ax{font-family:ui-monospace,SFMono-Regular,Cascadia Mono,Consolas,monospace;font-size:9.5px;fill:#8a8984}</style><text class="ti" x="24" y="28">Evidence waterfall &#8212; what each clue actually contributes</text><text class="su" x="24" y="46">Light track is the raw LLR from ln(m/u). Solid fill is what survives discounting and capping.</text><rect x="24" y="58" width="12" height="10" rx="2" fill="#eceae3"/><text class="no" x="42" y="67">raw LLR</text><rect x="112" y="58" width="12" height="10" rx="2" fill="#2a78d6"/><text class="no" x="130" y="67">counted contribution</text><text class="lb" x="24" y="99">PGP fingerprint exact</text><text class="mo" x="24" y="112" style="fill:#8a8984">EXACT_IDENTITY</text><rect x="236" y="88" width="372.0" height="20" rx="4" fill="#eceae3"/><path d="M236 88 h198.0 a4 4 0 0 1 4 4 v12 a4 4 0 0 1 -4 4 h-198.0 z" fill="#2a78d6"/><text class="vl" x="618.0" y="103">10.00</text><text class="no" x="666.0" y="103">capped at 10.0</text><text class="lb" x="24" y="143">BTC address reuse</text><text class="mo" x="24" y="156" style="fill:#8a8984">FINANCIAL</text><rect x="236" y="132" width="231.4" height="20" rx="4" fill="#eceae3"/><path d="M236 132 h122.4 a4 4 0 0 1 4 4 v12 a4 4 0 0 1 -4 4 h-122.4 z" fill="#2a78d6"/><text class="vl" x="477.4" y="147">6.26</text><text class="no" x="525.4" y="147">scaled to family cap 7.5</text><text class="lb" x="24" y="187">BTC co-input cluster</text><text class="mo" x="24" y="200" style="fill:#8a8984">FINANCIAL</text><rect x="236" y="176" width="184.0" height="20" rx="4" fill="#eceae3"/><path d="M236 176 h21.0 a4 4 0 0 1 4 4 v12 a4 4 0 0 1 -4 4 h-21.0 z" fill="#2a78d6"/><text class="vl" x="430.0" y="191">1.24</text><text class="no" x="478.0" y="191">lambda x 0.25, then cap</text><text class="lb" x="24" y="231">Favicon mmh3 match</text><text class="mo" x="24" y="244" style="fill:#8a8984">INFRASTRUCTURE</text><rect x="236" y="220" width="184.4" height="20" rx="4" fill="#eceae3"/><path d="M236 220 h97.0 a4 4 0 0 1 4 4 v12 a4 4 0 0 1 -4 4 h-97.0 z" fill="#2a78d6"/><text class="vl" x="430.4" y="235">5.00</text><text class="no" x="478.4" y="235">capped at 5.0</text><text class="lb" x="24" y="275">Stylometry (Burrows Delta)</text><text class="mo" x="24" y="288" style="fill:#8a8984">STYLOMETRY</text><rect x="236" y="264" width="89.1" height="20" rx="4" fill="#eceae3"/><path d="M236 264 h56.6 a4 4 0 0 1 4 4 v12 a4 4 0 0 1 -4 4 h-56.6 z" fill="#2a78d6"/><text class="vl" x="335.1" y="279">3.00</text><text class="no" x="383.1" y="279">capped at 3.0</text><text class="lb" x="24" y="319">Diurnal overlap</text><text class="mo" x="24" y="332" style="fill:#8a8984">TEMPORAL</text><rect x="236" y="308" width="39.4" height="20" rx="4" fill="#eceae3"/><path d="M236 308 h35.4 a4 4 0 0 1 4 4 v12 a4 4 0 0 1 -4 4 h-35.4 z" fill="#2a78d6"/><text class="vl" x="285.4" y="323">1.95</text><text class="no" x="333.4" y="323">counted in full</text><line x1="24" y1="358" x2="856" y2="358" stroke="#dedcd5"/><text class="lb" x="24" y="384" style="font-size:12.5px">Final LLR</text><text class="vl" x="236" y="384" style="font-size:14px;fill:#2a78d6">27.45</text><text class="no" x="310" y="384">raw sum was 54.48 &#8212; the engine discarded 27.03 as correlated or over-cap</text><text class="lb" x="24" y="410" style="font-size:12.5px">Decision</text><text class="vl" x="236" y="410" style="fill:#1baf7a">HIGH_CONFIDENCE_LINK</text><text class="no" x="422" y="410">posterior P &#8776; 1.0 &#183; 5 independent families agreeing</text></svg></div>

<div style="font-size:12px;color:#8a8984;font-family:ui-sans-serif,system-ui,sans-serif;margin-bottom:14px">Figure 4 &mdash; The same result as the analyst sees it in the UI. Every bar is traceable to a raw artifact.</div>

### 6.1 Family-level breakdown

The same result, grouped by family &mdash; this is what drives the stacked bar chart in
`EvidenceWaterfall.tsx`.

In [3]:
from packages.common.types import FAMILY_CAPS

print(f"{'FAMILY':<18} {'SCORE':>7} {'CAP':>6}  {'UTILISATION'}")
print("-" * 60)
for fam, score in result.family_scores.items():
    cap = FAMILY_CAPS[[f for f in FAMILY_CAPS if f.value == fam][0]]
    if score == 0:
        continue
    filled = int(round(score / cap * 24))
    bar = "#" * filled + "." * (24 - filled)
    print(f"{fam:<18} {score:>7.2f} {cap:>6.1f}  {bar} {score/cap*100:>5.1f}%")

print()
print("total capped LLR :", round(result.total_capped_llr, 2))

FAMILY               SCORE    CAP  UTILISATION
------------------------------------------------------------
EXACT_IDENTITY       10.00   10.0  ######################## 100.0%
FINANCIAL             7.50    7.5  ######################## 100.0%
INFRASTRUCTURE        5.00    5.0  ######################## 100.0%
STYLOMETRY            3.00    3.0  ######################## 100.0%
TEMPORAL              1.95    2.0  #######################.  97.3%

total capped LLR : 27.45


## 7. Three scenarios &mdash; the engine refusing to overclaim

The design goal is not a high score. It is a **defensible** score. These three cases show
the engine behaving correctly in the two situations that matter most: when evidence
conflicts, and when evidence is merely suggestive.

In [4]:
scenarios = {}

# A: strong, multi-family agreement (from above)
scenarios["A  strong multi-family"] = result

# B: identical evidence, plus one hard contradiction
with_conflict = evidence + [item("E7", "pgp_key_conflict")]
scenarios["B  same, one contradiction"] = evaluate_attribution(with_conflict)

# C: only weak, circumstantial signals
weak = [item("W1", "handle_trigram_fuzzy"), item("W2", "temporal_diurnal_fit")]
scenarios["C  weak circumstantial"] = evaluate_attribution(weak)

print(f"{'SCENARIO':<28} {'LLR':>8} {'PENALTY':>9} {'P':>10}  DECISION")
print("-" * 84)
for name, r in scenarios.items():
    print(f"{name:<28} {r.final_llr:>8.2f} {r.contradiction_penalty:>9.1f} "
          f"{r.posterior_probability:>10.4f}  {r.decision.value}")

SCENARIO                          LLR   PENALTY          P  DECISION
------------------------------------------------------------------------------------
A  strong multi-family          27.45       0.0     1.0000  HIGH_CONFIDENCE_LINK
B  same, one contradiction       7.45      20.0     0.9957  CONTRADICTION_REJECTED
C  weak circumstantial           3.95       0.0     0.8750  HIGH_CONFIDENCE_LINK


**Scenario B is the important one.** The positive evidence was overwhelming &mdash; a
PGP key match, a wallet, infrastructure, stylometry. One conflicting PGP key drags the LLR
from 27.45 down to 7.45 and flips the decision to `CONTRADICTION_REJECTED`. The system
refuses to name an actor it cannot cleanly defend.

> **Known issue worth naming.** In Scenario B the posterior still reads 0.9957 even though
> the decision is a rejection, because the contradiction short-circuits the ladder *after*
> the probability is computed. The decision is right; the number displayed beside it is
> misleading. Scenario C has the mirror problem &mdash; two weak signals reach
> `HIGH_CONFIDENCE_LINK` at P = 0.875, because the sigmoid prior of &minus;2.0 is too
> generous when only weak families are present. **Both are calibration bugs, not
> architecture bugs**, and both are exactly what fitting the isotonic calibrator on the
> `bench/` ground-truth set is meant to fix in Phase 1.

## 8. Stylometry

`packages/stylometry` implements same-author verification on three feature groups:

| Feature | Function |
|---|---|
| **Character n-grams** (3&ndash;5) | Sub-word habits that survive vocabulary change |
| **Function-word frequencies** | `the`, `of`, `but` &mdash; used unconsciously, hard to fake |
| **Punctuation ratios** | Per-100-character rates of each mark |
| **Sentence distributions** | Length mean and variance |

The module enforces a **hard abstention rule: fewer than 50 words and it refuses to
score.** This follows Narayanan et al., *"On the Feasibility of Internet-Scale Author
Identification"* (IEEE S&amp;P 2012), whose key finding is that precision rises from ~20%
to **over 80%** when a model is permitted to abstain rather than always guess.

Abstention is why stylometry is capped at 3.0 rather than excluded: it is a real signal,
it is adversarially defeatable, and the engine treats it accordingly.

In [5]:
from packages.stylometry.episodes import StylometryEpisode, MIN_WORD_COUNT_THRESHOLD

long_post = (
    "Vendor has been reliable across three separate transactions now. Shipping was "
    "discreet, packaging vacuum sealed, and the stealth was honestly better than "
    "advertised on the listing page. Would recommend to anyone asking, though do your "
    "own diligence as always. Communication was prompt throughout and the escrow "
    "process went through without a single issue on either side of the trade."
)
short_post = "Fast shipping, good stealth, will order again."

print("abstention threshold:", MIN_WORD_COUNT_THRESHOLD, "words\n")

for label, text in [("long post", long_post), ("short post", short_post)]:
    ep = StylometryEpisode.from_single_text("actor_a", label, text)
    print(f"{label:<12} words={ep.word_count:<4} abstain={str(ep.abstain):<6}", end="")
    if ep.abstain:
        print("-> emits 0.0 weight, no score claimed")
    else:
        f = ep.feature_dict
        print(f"-> {len(f['function_word_vector'])} function words, "
              f"{len(f['char_ngrams'])} char n-grams")

print()
print("punctuation ratios of the scored episode (per 100 chars):")
scored = StylometryEpisode.from_single_text("actor_a", "long", long_post)
for mark, ratio in list(scored.feature_dict["punctuation_ratios"].items())[:5]:
    print(f"   {mark:<10} {ratio:>7.3f}")

abstention threshold: 50 words

long post    words=59   abstain=False -> 174 function words, 999 char n-grams
short post   words=7    abstain=True  -> emits 0.0 weight, no score claimed

punctuation ratios of the scored episode (per 100 chars):
   punct_.      1.044
   punct_,      0.783
   punct_!      0.000
   punct_?      0.000
   punct_;      0.000


## 9. API and product surface

### REST API &mdash; 24 routes in `apps/api/main.py`

| Group | Routes |
|---|---|
| **Auth** | `POST /api/v1/auth/login`, `GET /api/v1/auth/me` &mdash; JWT bearer |
| **Actors** | list, detail, `/graph` (Cytoscape-shaped), `/timeline` |
| **Evidence** | list, detail *(read-only &mdash; no ingestion endpoint yet)* |
| **Hypotheses** | list, detail with waterfall, `POST /review` |
| **Attribution** | `POST /api/v1/attribution/evaluate` &mdash; the engine, live |
| **Investigations** | list, create &mdash; case management |
| **Search** | `GET /api/v1/search` &mdash; handle, wallet, PGP fingerprint |
| **Exports** | `report` (PDF), `json`, `stix` (STIX 2.1), `csv` |
| **Audit** | `GET /api/v1/audit`, `GET /api/v1/audit/verify` |
| **Copilot** | `POST /api/v1/copilot/query` |

### Tamper-evident audit chain

Every state change appends a record whose hash includes its predecessor's hash &mdash; a
SHA-256 hash chain. `GET /api/v1/audit/verify` walks the whole chain and reports whether
any record was altered after the fact. For a system that produces evidence intended to
support real-world action, **the audit log is not a feature, it is the credibility
mechanism.**

### Frontend &mdash; 13 components in `apps/web/src/components`

`ReviewQueue` (hypotheses ranked by calibrated probability) &middot; `EvidenceWaterfall`
(stacked family contributions, contradictions pulling left in red, drill-down to raw
provenance) &middot; `GraphExplorer` (Cytoscape, edge thickness proportional to LLR
contribution) &middot; `AttributionLab` (accept / reject / insufficient, plus exports)
&middot; `ActorProfile` &middot; `CasesView` &middot; `EvidenceVault` &middot;
`AuditLogViewer` &middot; `CopilotDrawer` &middot; `CommandCenter` &middot; `AppShell`
&middot; `LoginScreen`

## 10. Verification report

Audited against the working tree on 30 August 2026. This section exists so nobody &mdash;
teammate or judge &mdash; is surprised by a gap we already knew about.

### Test suite

`pytest` &rarr; **38 passed, 1 failed** (`test_hash_chained_audit_integrity`, a SQLAlchemy
error). Fix before any demo; a red audit-chain test undercuts the exact claim &sect;9 makes.

### Blocking defects

| Defect | Location | Impact |
|---|---|---|
| `Any` used but never imported | `workers/extraction/extractor.py:25` | **Module raises `NameError` on import.** Kills the PGP, BTC/ETH/XMR, and favicon extractors. |
| `Any` used but never imported | `workers/extraction/clone_detector.py` | **Same.** Kills the SimHash clone detector. |
| `pyyaml` imported, never declared | `fusion.py:15` vs `pyproject.toml` | A clean `pip install -e .` yields a broken fusion engine. |
| Build path does not exist | `docker-compose.yml` &rarr; `infrastructure/docker/Dockerfile.api` | `docker compose up` cannot build the API. |

Both `Any` bugs are one-line fixes. Neither is caught by the test suite, because nothing
outside `workers/` imports either module &mdash; they are currently dead code.

### Slide vs. repository

| Slide claim | Reality | Action |
|---|---|---|
| GitHub Actions | No `.github/` directory; not in any roadmap phase | **Remove from slide** (or add a CI file) |
| SSDEEP | Only SimHash exists; SSDEEP is in no phase | **Remove the word** (or add `ppdeep`) |
| Isotonic calibration | Sigmoid is the live path | **Relabel** |
| Evidence Ingestion APIs | All evidence routes are read-only | Relabel as *Query &amp; Retrieval*, or build `POST` |
| MinIO, Redis | Genuinely Phase 2 &mdash; not overclaims | Mark as planned, keep |

### Built but *missing* from the slide

`STIX 2.1 export` &middot; `SHA-256 hash-chained audit` &middot; `AI Copilot` &middot;
`pgvector` &middot; `bench/` calibration metrics (ECE, Brier, FAR, Recall@10).

These are finished work getting no credit. STIX 2.1 in particular signals to a
defence-sector evaluator that the system speaks standard CTI interchange.

## 11. Roadmap

From [`docs/ROADMAP.md`](docs/ROADMAP.md).

| Phase | Scope | State |
|---|---|---|
| **0** | Stabilize MVP &mdash; FastAPI + Next.js, UUIDv7 keys, audit chain, compose mesh | Complete |
| **1** | Attribution engine &mdash; LLR fusion, lambda discount, family caps, isotonic calibration, stylometry | **Active** |
| **2** | Passive collection &mdash; WARC to MinIO, onion probes, Redis Streams bus, extractors | **Active** |
| **3** | Real-time frontend &mdash; review queue, waterfall, attribution lab, graph explorer | **Active** |
| **4** | Hardening &mdash; object-level RBAC, OpenSearch indexing, Prometheus/Grafana | Upcoming |
| **5** | Advanced &mdash; RoBERTa/DeBERTa stylometry, cross-ledger clustering, RAG copilot | Future |

**Phase 1 definition of done:** ECE &lt; 0.15, Brier &lt; 0.08, false-attribution rate
0.0%, Recall@10 100% &mdash; measured by `bench/report.py`. Fitting the isotonic calibrator
against this benchmark is what closes the calibration gaps noted in &sect;7.

## 12. Research foundations

The design is grounded in published work, not invented. Cite these directly if challenged.

| Source | Role in NETRA-X |
|---|---|
| **Fellegi &amp; Sunter (1969)**, probabilistic record linkage | The formal model behind the whole fusion engine |
| **Narayanan, Paskov et al. (IEEE S&amp;P 2012)**, *Internet-Scale Author Identification* | Abstention design; precision ~20% &rarr; &gt;80% when the model may decline |
| **Narayanan &amp; Shmatikov (IEEE S&amp;P 2009)**, *De-anonymizing Social Networks* | Seed-and-propagate from a confirmed identity through a trust graph |
| **OnionScan** | Hidden-service misconfiguration scanning &mdash; wrapped, not rebuilt |
| **TorBot** | Dark-web OSINT crawling for ingestion |
| **VeriDark** (bit-ml) | Real dark-web authorship datasets for stylometry training |
| **NATO Admiralty System** | A&ndash;F / 1&ndash;6 source-reliability grading, rather than an invented scheme |
| **Have I Been Pwned** | Breach **membership** checking &mdash; membership only, never credentials |
| **ISO 28500** | WARC record format for immutable capture |

### VeriDark access note

Only `MiniDarkReddit` (~200&ndash;400 samples per split) is publicly downloadable. The
larger sets (DarkReddit+, SilkRoad1, Agora) need a Zenodo access request with an
institutional email &mdash; likely too slow for hackathon timelines. Plan around
MiniDarkReddit.

VeriDark's published ethics policy explicitly forbids using the dataset to unmask
undercover agents, journalists, dissidents, or whistleblowers. **Cite this in the pitch**
&mdash; it reinforces rather than conflicts with the guardrails below.

## 13. Ethical and legal guardrails

**These are non-negotiable and must not be weakened.** They are the line between an
attribution tool and an unauthorized surveillance system.

1. **No live scraping of real darknet marketplaces**, ever, in build or demo. Ingestion is
   demoed against VeriDark and public academic research corpora.
   *Disclosure line for the pitch:* &ldquo;Ingestion is demoed against a public research
   corpus; production would plug into NTRO's own authorized collection pipeline.&rdquo;

2. **Face matching never auto-confirms an identity.** Detection and embedding comparison
   may run automatically, but a match only ever reaches `flagged_for_human_review`. This is
   enforced at the schema level &mdash; the `face_match_status` enum has **no** automatic
   confirmed state.

3. **Breach data: membership only, never credentials.** The system records which breaches
   an email appeared in. It never stores a leaked password, hashed or plaintext. Doing so
   would cross from attribution into possession of stolen data.

4. **Seed-and-propagate results are always flagged as inference**, never presented at the
   same confidence tier as direct evidence. Proximity to a caught associate is a lead, not
   proof.

5. **The system matches against real-identity records an investigator supplies. It does not
   go out and build profiles on private individuals on its own.**

6. **The test onion service is self-hosted and isolated.** Module 1 is only ever pointed at
   the team's own instance with deliberately planted misconfigurations &mdash; never at a
   real, live hidden service.

### Why this is a competitive advantage, not a constraint

Every one of these is a line a careless team will cross and get questioned on. Stating them
first, unprompted, signals to a government-sponsored evaluator that the team understands
the difference between a tool NTRO could actually deploy and a liability.

## 14. Team

Three pairs, each owning a coherent domain across every build stage. Nobody works alone;
every pair has a built-in review partner.

| Pair | Domain | Members |
|---|---|---|
| **A** | Core Backend &mdash; data store, graph engine, fusion and scoring | Krishna, Chaitanya |
| **B** | Signal Intelligence &mdash; extraction, stylometry, behavioral modeling | Varsharani, Sakshi |
| **C** | Product &amp; Live Systems &mdash; infra fingerprinting, dashboard, integration, demo | Vivek, Sahil |

---

## Immediate next actions

1. Fix the two `Any` imports &mdash; two lines, restores four extraction modules
2. Add `pyyaml` to `pyproject.toml`
3. Fix `test_hash_chained_audit_integrity`
4. Point `docker-compose.yml` at the root `Dockerfile`
5. Correct the slide: remove GitHub Actions and SSDEEP, relabel calibration as sigmoid,
   add STIX 2.1 and the audit chain
6. Fit the isotonic calibrator on `bench/` ground truth &mdash; closes both calibration
   gaps from &sect;7 and completes Phase 1

---

<div style="font-size:12px;color:#8a8984;font-family:ui-sans-serif,system-ui,sans-serif">
Generated 30 August 2026 &middot; all engine figures produced by executing this repository
at commit <code>fb8067b</code>.
</div>